# Profiling

This notebok provides and understanding on
- Profiling CPU and memory usage
- Guidelines on depth of profiling
- Profiling long running apps

In [9]:
import random, time, cProfile, pstats, io


def work(n):
    data = []
    for _ in range(n):
        data.append(random.randint(1, 100))

    total = 0
    for x in data:
        total += x * x

    return total


pr = cProfile.Profile()
pr.enable()

result = work(1_000_000)

pr.disable()

s = io.StringIO()
pstats.Stats(pr, stream=s).sort_stats("tottime").print_stats(5)

print("Result:", result)
print(s.getvalue())


Result: 3386605069
         9280270 function calls (9280263 primitive calls) in 1.672 seconds

   Ordered by: internal time
   List reduced from 137 to 5 due to restriction <5>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
  1000000    0.456    0.000    1.062    0.000 /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/random.py:295(randrange)
  1000000    0.295    0.000    0.449    0.000 /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/random.py:245(_randbelow_with_getrandbits)
        1    0.238    0.238    1.373    1.373 /var/folders/z5/b0wfp_7s5292vp_mpklvrn2w0000gn/T/ipykernel_12813/1496816697.py:4(work)
  1000000    0.174    0.000    1.236    0.000 /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/random.py:336(randint)
  3000000    0.157    0.000    0.157    0.000 {built-in method _operator.index}





ncalls – number of times the function was called

tottime – time spent inside the function (excluding subcalls)

cumtime – time spent in the function including subcalls

percall – average per call

filename:lineno(function) – where it lives

_________

## cProfile
**cProfile** is one if the two profilers in standard library, alongside profile.

profile is the original slower Python profiler while cProfile in written in C for lower overhead.

---------

## line_profiler

line_profiler works by profiling functions on a line-by-line basis, so you start by cProfile and use the high level view to guide which functions to profile with line_profiler

Step 1: To install line_profiler, use the command pip install line_profiler

Step 2: load the extension `%load_ext line_profiler`

Step 3: run profiling on your function

_Restart your kernel for the installation to take effect_

In [3]:
%load_ext line_profiler

In [4]:
%lsmagic

Available line magics:
%alias  %alias_magic  %autoawait  %autocall  %automagic  %autosave  %bookmark  %cat  %cd  %clear  %code_wrap  %colors  %conda  %config  %connect_info  %cp  %debug  %dhist  %dirs  %doctest_mode  %ed  %edit  %env  %gui  %hist  %history  %killbgscripts  %ldir  %less  %lf  %lk  %ll  %load  %load_ext  %loadpy  %logoff  %logon  %logstart  %logstate  %logstop  %lprun  %ls  %lsmagic  %lx  %macro  %magic  %mamba  %man  %matplotlib  %micromamba  %mkdir  %more  %mv  %notebook  %page  %pastebin  %pdb  %pdef  %pdoc  %pfile  %pinfo  %pinfo2  %pip  %popd  %pprint  %precision  %prun  %psearch  %psource  %pushd  %pwd  %pycat  %pylab  %qtconsole  %quickref  %recall  %rehashx  %reload_ext  %rep  %rerun  %reset  %reset_selective  %rm  %rmdir  %run  %save  %sc  %set_env  %store  %subshell  %sx  %system  %tb  %time  %timeit  %unalias  %unload_ext  %uv  %who  %who_ls  %whos  %xdel  %xmode

Available cell magics:
%%!  %%HTML  %%SVG  %%bash  %%capture  %%code_wrap  %%debug  %%file  %%htm

In [5]:
%lprun -f work work(1_000_000)

Timer unit: 1e-09 s

Total time: 3.63773 s
File: /var/folders/z5/b0wfp_7s5292vp_mpklvrn2w0000gn/T/ipykernel_90446/1496816697.py
Function: work at line 4

Line #      Hits         Time  Per Hit   % Time  Line Contents
     4                                           def work(n):
     5         1          0.0      0.0      0.0      data = []
     6   1000001  176007000.0    176.0      4.8      for _ in range(n):
     7   1000000 3142888000.0   3142.9     86.4          data.append(random.randint(1, 100))
     8                                           
     9         1          0.0      0.0      0.0      total = 0
    10   1000001  146391000.0    146.4      4.0      for x in data:
    11   1000000  172440000.0    172.4      4.7          total += x * x
    12                                           
    13         1       1000.0   1000.0      0.0      return total

## The big conclusion (key learning)

From the profiler:

| Component                  | % Time |
|----------------------------|--------|
| Random number generation   | 86.4%  |
| Loop overhead              | ~4.8%  |
| Arithmetic                 | ~4.7%  |
| List iteration             | ~4.0%  |


If you want this code to be faster, optimizing loops or arithmetic will barely help.
You must reduce or replace calls to `random.randint()`.


## memory_profiler

`memory_profiler` is a Python tool used to measure how much **memory (RAM)** a program uses during execution. It can show memory usage **line by line**, helping developers identify parts of the code that allocate or consume large amounts of memory.

`memory_profiler` operates similarly as `line_profiler` by profiling code line by line, but it runs significantly more slowly due to the overhead of tracking memory usage.

### When it can be used
`memory_profiler` is useful when:
- A program is using **too much memory**.
- You suspect a **memory leak**.
- You want to understand **which lines of code allocate large objects**.
- Optimizing applications that process **large datasets** (e.g., data processing, machine learning, ETL pipelines).

By identifying memory-heavy lines of code, developers can refactor or optimize the program to reduce memory consumption.

To install do a `pip install memory-profiler`

Add `from memory_profiler import profile`

In [10]:
%load_ext memory_profiler

The memory_profiler extension is already loaded. To reload it, use:
  %reload_ext memory_profiler


In [11]:
%mprun -f work work(1_000_000)

ERROR: Could not find file /var/folders/z5/b0wfp_7s5292vp_mpklvrn2w0000gn/T/ipykernel_12813/1496816697.py



## Comparison of Python Profiling Tools

| Tool | What it Measures | Granularity | Typical Use Case |
|-----|------------------|-------------|------------------|
| `cProfile` | Execution time of functions | Function-level | Identify which functions take the most runtime in a program |
| `line_profiler` | Execution time of each line in a function | Line-level | Find the exact lines of code that cause performance bottlenecks |
| `memory_profiler` | Memory (RAM) usage during execution | Line-level | Detect memory-heavy operations and potential memory leaks |

### Summary
- `cProfile` is useful for getting a **high-level overview of program performance**.
- `line_profiler` provides **detailed timing information for each line of code**.
- `memory_profiler` helps analyze **memory consumption**, but it runs slower due to the overhead of monitoring memory usage.

In [2]:
!pip install memory-profiler


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## py-spy

`py-spy` is a **sampling profiler** for Python that allows you to analyze CPU performance **without modifying your code**.
It runs **outside the Python process**, so it has very low overhead and can even profile **running programs**.

### Key Features

- **Sampling profiler**: periodically checks where your program is spending time
- **Low overhead**: minimal impact on performance
- **No code changes required**: works on existing scripts or running processes
- **Flame graph visualization**: generates interactive charts showing CPU hotspots

### When to Use

- Profiling **running programs**, including production scripts
- Diagnosing **CPU bottlenecks**
- Generating **flame graphs** for performance analysis
- Quickly identifying **slow functions** without instrumenting code

### Installation

```bash
pip install py-spy